# Create Persian dataset
After `download_data.py` and `filter_persian.py` are run, this notebook builds `train/` and `eval/` in `data/persian/` from `data/persian/raw/`. It samples images from `data/persian/raw/`, captions them with BLIP, removes caption prefixes, and removes screened-out junk images.

## Environment/setup

In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# clone repo
!git clone https://github.com/HarshaSrirangam/stable-diffusion-rebuilt.git
%cd stable-diffusion-rebuilt
!git pull

Cloning into 'stable-diffusion-rebuilt'...
remote: Enumerating objects: 429, done.
remote: Counting objects: 100% (429/429), done.
remote: Compressing objects: 100% (246/246), done.
remote: Total 429 (delta 225), reused 342 (delta 138), pack-reused 0 (from 0)
Receiving objects: 100% (429/429), 4.01 MiB | 7.87 MiB/s, done.
Resolving deltas: 100% (225/225), done.
/content/stable-diffusion-rebuilt/stable-diffusion-rebuilt
Already up to date.


In [4]:
# install dependencies (skip torch/numpy to avoid overwriting Colab's preinstalled packages)
%pip install -e . --no-deps -q
%pip install transformers safetensors tqdm accelerate pyyaml datasets -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for stable-diffusion-rebuilt (pyproject.toml) ... done


In [5]:
# mount drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
# symlink colab data/persian/ -> drive data/persian
!mkdir -p data # create colab data/
!ln -sfn /content/drive/MyDrive/sd-rebuilt/data/persian data/persian # colab data/persian points to drive persian
# build colab data/persian/train/

# Sample and BLIP caption train and eval splits

In [7]:
import re, json
from pathlib import Path

# helpers
STRUCT = [
    "a picture taken from a manuscript of", "a picture taken from",
    "a page of a manuscript with a painting of", "a page of a manuscript with a",
    "a page of a manuscript with", "a page of a manuscript",
    "page of a manuscript with a painting of", "page of a manuscript with a",
    "page of a manuscript with", "page of a manuscript",
    "a page of a book with a painting of", "a page of a book with a",
    "a page of a book with", "a page of a book",
    "page of a book with a", "page of a book", "a page of", "page of",
    "a book with a picture of", "a book with a painting of", "a book with a drawing of",
    "a book with a", "a book with", "a book",
    "a persian manuscript with a painting of", "a persian manuscript with a",
    "a persian manuscript with", "a persian manuscript",
    "a manuscript with a painting of", "a manuscript with a", "a manuscript with",
    "a manuscript of", "a manuscript",
    "a close up of", "close up of", "there is", "a scene of",
]
STRUCT.sort(key=len, reverse=True)

GEN = re.compile(
    r"^(a |an |the )?(black and white )?(\w+ ){0,3}"
    r"(painting|drawing|picture|image|illustration|photo|portrait|depiction|artwork) of "
)

def clean(text):
    t = text.lower().strip()
    t = re.sub(r"\barafed\b|\baraffe\b|\baraf\b", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    changed = True
    while changed:
        changed = False
        for p in STRUCT:
            if t == p or t.startswith(p + " "):
                t = t[len(p):].strip(); changed = True; break
        m = GEN.match(t)
        if m:
            t = t[m.end():].strip(); changed = True
    t = re.sub(r"\s+", " ", t).strip(" ,.")
    return t if len(t.split()) >= 2 else "a painting"


# QC methods
def clean_captions(split):
    root = Path("data/persian") / split
    path = root / "metadata.jsonl"
    raw = root / "metadata_raw.jsonl"
    if not raw.exists():   # back up original BLIP captions
        path.rename(raw)
    with open(raw) as f, open(path, "w") as out:
        for line in f:
            row = json.loads(line)
            row["text"] = clean(row["text"])
            out.write(json.dumps(row) + "\n")
    print(f"cleaned -> {path}")


def remove_junk(split, reject):
    root = Path("data/persian") / split
    imgs = root / "images"
    meta = root / "metadata.jsonl"
    reject_names = {f"images/{i:04d}.jpg" for i in reject}
    for i in reject:
        p = imgs / f"{i:04d}.jpg"
        if p.exists():
            p.unlink()
    lines = [l for l in open(meta) if json.loads(l)["file_name"] not in reject_names]
    with open(meta, "w") as f:
        f.writelines(lines)
    print(f"removed {len(reject)}; remaining {len(lines)}")

In [8]:
from pathlib import Path
import shutil
import random

raw_dir = Path("data/persian/raw")
TRAIN_SIZE = 1300 # more than 1200 to account for junk removal
EVAL_SIZE = 150
seed = 42

# sample TRAIN_SIZE + EVAL_SIZE images from raw/, then split
files = sorted(raw_dir.glob("*.jpg"))
sample = random.Random(seed).sample(files, TRAIN_SIZE + EVAL_SIZE)
splits = {"train": sample[:TRAIN_SIZE], "eval": sample[TRAIN_SIZE:]}

for split, split_files in splits.items():
    out_dir = Path("data/persian") / split / "images"
    if out_dir.exists() and any(out_dir.glob("*.jpg")):
        print(f"{split} already populated ({len(list(out_dir.glob('*.jpg')))}), skipping")
        continue
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, img_path in enumerate(split_files):
        shutil.copy(img_path, out_dir / f"{i:04d}.jpg")
    print(f"{split}: copied {len(split_files)} images to {out_dir}")

train: copied 1300 images to data/persian/train/images
eval: copied 150 images to data/persian/eval/images


In [9]:
import json
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

def caption_split(model, processor, device, split):
    root = Path("data/persian") / split
    if (root / "metadata.jsonl").exists():
        print(f"{split}/metadata.jsonl exists, skipping")
        return
    paths = sorted((root / "images").glob("*.jpg"))
    B = 16
    with open(root / "metadata.jsonl", "w") as meta:
        for i in tqdm(range(0, len(paths), B), desc=split):
            chunk = paths[i:i+B]
            imgs = [Image.open(p).convert("RGB") for p in chunk]
            inp = processor(images=imgs, return_tensors="pt").to(device, torch.float16)
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=30)
            caps = processor.batch_decode(out, skip_special_tokens=True)
            for p, cap in zip(chunk, caps, strict=True):
                meta.write(json.dumps({"file_name": f"images/{p.name}", "text": cap.strip()}) + "\n")

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large", torch_dtype=torch.float16
).to(device).eval()

for split in ("train", "eval"):
    caption_split(model, processor, device, split)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.60k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

eval: 100%|██████████| 10/10 [00:12<00:00,  1.20s/it]


In [10]:
# remove prefixes from captions
clean_captions("train")
clean_captions("eval")

cleaned -> data/persian/train/metadata.jsonl
cleaned -> data/persian/eval/metadata.jsonl


In [11]:
# remove junk after manually screening images
remove_junk("train", [
    14, 16, 24, 74, 81, 84, 110, 136, 140, 183, 212, 220, 240, 259, 317,
    345, 347, 360, 386, 407, 483, 485, 500, 567, 584, 613, 614, 651, 688,
    703, 708, 746, 772, 779, 785, 923, 925, 929, 951, 959, 981, 983, 1018,
    1030, 1063, 1065, 1086, 1111, 1117, 1196, 1217, 1218, 1245, 1281,
])
remove_junk("eval", [29, 38, 46, 48, 59, 74, 111, 142])

removed 54; remaining 1246
removed 8; remaining 142


In [12]:
# verify counts
!ls data/persian/train/images | wc -l
!ls data/persian/eval/images | wc -l

1246
142
